<a href="https://colab.research.google.com/github/LiuChen-5749342/Generative-AI-and-AI-Applications/blob/main/Task_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Task 16**

Add a new tool called get_pe_ratio that fetches the Price-to-Earnings ratio. Then ask the agent if Apple is 'overvalued' compared to the average market P/E of 25.

Hint:

stock = yf.Ticker(ticker)

PE ratio is often in 'summaryDetail' or 'info'

pe = stock.info.get('trailingPE', 'N/A')

**Setup and API Configuration**

This cell handles the initial setup. It installs the necessary `google-generativeai` library, imports it, and then configures the Gemini API using your API key. The API key is securely retrieved from Colab's user data secrets.

In [2]:
# Setup (Use the standard library)
# Get your API key from https://aistudio.google.com/
!pip install -q -U google-generativeai # Ensure the correct library is installed
import google.generativeai as genai # Import the correct module to override previous
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

In [3]:
!pip install -q yfinance

### Define Custom Tools (Python Functions)
Here, we define two Python functions that will act as 'tools' for our Gemini agent:

1.  **`get_stock_price(ticker)`**: Takes a stock ticker symbol (e.g., 'AAPL') and returns its current live price.
2.  **`get_company_risk_score(ticker)`**: Calculates a simplified risk assessment for a given stock based on its Beta value, which measures market volatility. It categorizes risk as 'High', 'Moderate', or 'Low'.
3. **get_pe_ratio**: fetches the Price-to-Earnings ratio.

In [4]:
import yfinance as yf

# Define the "Real" Tools

def get_stock_price(ticker: str):
    """
    Retrieves the current live stock price for a given ticker.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'NVDA').
    """
    print(f"  ... 🔍 TOOL CALL: Connecting to Yahoo Finance for {ticker} ...")

    try:
        stock = yf.Ticker(ticker)
        price = stock.fast_info['last_price']
        return round(price, 2)
    except Exception as e:
        return f"Error fetching price for {ticker}: {e}"


def get_company_risk_score(ticker: str):
    """
    Calculates a risk proxy based on the stock's Beta (market volatility).
    A Beta > 1.0 means higher risk/volatility than the market.
    Args:
        ticker: The stock ticker symbol.
    """
    print(f"  ... ⚠️ TOOL CALL: Fetching Risk Metrics for {ticker} ...")

    try:
        stock = yf.Ticker(ticker)
        beta = stock.info.get('beta', 0)

        if beta > 1.5:
            assessment = "High Risk (High Volatility)"
        elif beta < 0.8:
            assessment = "Low Risk (Stable)"
        else:
            assessment = "Moderate Risk"

        return {"beta": beta, "assessment": assessment}
    except Exception as e:
        return "Risk data unavailable"


def get_pe_ratio(ticker: str):
    """
    Retrieves the trailing Price-to-Earnings (P/E) ratio for a given stock.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'MSFT').
    """
    print(f"  ... 📊 TOOL CALL: Fetching P/E Ratio for {ticker} ...")

    try:
        stock = yf.Ticker(ticker)
        pe = stock.info.get('trailingPE', None)

        if pe is None:
            return f"P/E ratio unavailable for {ticker}"

        return round(pe, 2)
    except Exception as e:
        return f"Error fetching P/E ratio for {ticker}: {e}"


print("✅ Real-Time Tools Defined.")

✅ Real-Time Tools Defined.


In [9]:
# Initialize Model with Tools
# We use 'gemini-2.5-flash' as it is fast and excellent at tool use.

tools_list = [get_stock_price, get_company_risk_score, get_pe_ratio]

model = genai.GenerativeModel(
    "gemini-2.5-flash",
    tools=tools_list
)

# Enable automatic function calling
# This tells the SDK: "If the model asks to run a function, just run it for me and give the answer back."
chat = model.start_chat(enable_automatic_function_calling=True)

print("✅ Model initialized with Agentic capabilities.")

✅ Model initialized with Agentic capabilities.


In [10]:
# Test: Single Tool Call
response = chat.send_message("Is Apple 'overvalued' compared to the average market P/E of 25?")

print("\n🤖 AGENT RESPONSE:")
print(response.text)

  ... 📊 TOOL CALL: Fetching P/E Ratio for AAPL ...

🤖 AGENT RESPONSE:
Apple's P/E ratio is 31.35, which is higher than the average market P/E of 25. This suggests that Apple might be overvalued compared to the average market.
